# stray_model_v4 — เทรนใหม่ทั้งชุด (สูตรใหม่ แก้ปัญหาจาก v3)

**สิ่งที่เปลี่ยนจาก v3 (เพื่อแก้ปัญหา Ragdoll/Birman สับสน + ความแม่นรวมตก):**

| หัวข้อ | v3 (เดิม) | v4 (ใหม่) | เหตุผล |
|---|---|---|---|
| Oversampling | ทำให้คลาสน้อยเท่าคลาสมาก (เต็มขั้น) | จำกัดไว้ไม่เกิน 50% ของคลาสที่มากสุด | ลดการซ้ำภาพมากเกินไป ซึ่งพอผสมกับ MixUp แล้วทำให้ภาพสายพันธุ์หายาก "เบลอ" กว่าเดิม |
| MixUp | ใช้ทุก Phase (1-2-3) | ใช้เฉพาะ Phase 1 (ตอน backbone ยัง freeze) ปิดใน Phase 2-3 | MixUp ตอน fine-tune ชั้นลึกจะไปเบลอจุดต่างเล็กๆ ระหว่างสายพันธุ์หน้าคล้ายกัน (เช่น Ragdoll/Birman) — Phase 2-3 คือช่วงที่โมเดลควรเรียนจุดต่างละเอียดพวกนี้ |
| Focal Loss gamma | 2.0 | 1.5 + label smoothing 0.05 | โฟกัสตัวอย่างยากน้อยลงหน่อย กันโมเดล over-focus จนสร้าง bias ใหม่ |
| Class weight | ไม่มี (ใช้ oversampling อย่างเดียว) | มีเพิ่ม (คู่กับ oversampling แบบเบา) | กระจายการแก้ class imbalance เป็น 2 ทางที่เบากว่าทางเดียวหนักๆ |
| Early stopping | ไม่มี | มี (monitor val_macro_f1, patience=5) | กัน Phase 3 เทรนนานเกินจนเริ่ม overfit ทับจุดที่ดีที่สุดไปแล้ว |
| กราฟ/ตารางสรุปผล | ไม่มีในโน้ตบุ๊ก (ทำแยก) | มีในตัวครบ: Accuracy/Loss per epoch รวม 3 phase, ตาราง Precision/Recall/F1 ทุกสายพันธุ์, Confusion Matrix (ดิบ + normalize) | ตามที่ขอ |

**วิธีใช้:** รันทีละ cell ตามลำดับ ห้ามข้าม แต่ละ cell มีคำอธิบาย "ควรได้ผลลัพธ์ประมาณไหน" กำกับไว้ด้านบน ถ้ารันแล้ว error หรือผลลัพธ์ไม่ตรง — ส่ง screenshot กลับมาได้เลย


## Cell 1 — ตั้งค่าเริ่มต้น + เช็ค GPU

**ควรได้:** ข้อความ `Mounted at /content/drive` และ `GPU: /device:GPU:0` (ถ้าขึ้น `GPU: ไม่พบ` ให้ไปที่ Runtime → Change runtime type → เลือก GPU (T4) ก่อนรันต่อ)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpu = tf.config.list_physical_devices('GPU')
print("GPU:", gpu[0].name if gpu else "ไม่พบ (ไปเปลี่ยน Runtime เป็น GPU ก่อน)")


## Cell 2 — ตั้งค่าฟอนต์ไทยสำหรับกราฟ (กัน matplotlib ขึ้นเป็นกล่องสี่เหลี่ยม)

**ควรได้:** ข้อความ `ตั้งค่าฟอนต์ไทยสำเร็จ...` ไม่มี error (ใช้เวลาสักครู่ตอน apt-get ติดตั้งฟอนต์) ต้องรัน cell นี้ก่อนกราฟทุกอันในโน้ตบุ๊ก ไม่งั้นตัวอักษรไทยในกราฟ (title/label) จะขึ้นเป็นสี่เหลี่ยมว่างๆ แทน

In [ ]:
!apt-get -qq install -y fonts-thai-tlwg > /dev/null 2>&1

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import glob

for font_file in glob.glob('/usr/share/fonts/**/*.ttf', recursive=True):
    if any(name in font_file.lower() for name in ['loma', 'garuda', 'norasi', 'kinnari', 'sawasdee', 'tlwg']):
        fm.fontManager.addfont(font_file)

plt.rcParams['font.family'] = 'Loma'
plt.rcParams['axes.unicode_minus'] = False

print("ตั้งค่าฟอนต์ไทยสำเร็จ กราฟหลังจากนี้จะอ่านตัวอักษรไทยได้ปกติ")


## Cell 3 — ตั้งค่าพารามิเตอร์หลัก (Config)

**ควรได้:** ไม่มี error พิมพ์ค่า config ออกมาให้ดู **จุดสำคัญ:** `DATA_DIR` ใช้ดิสก์ของ Colab เอง (`/content/...`) ไม่ใช่ Drive เพราะเขียนไฟล์รูปหลายพันไฟล์ลง Google Drive ที่ mount ผ่าน FUSE ช้ามาก (เป็นชั่วโมง) ส่วน `SAVE_PATH` (ผลลัพธ์สุดท้าย มีแค่ไม่กี่ไฟล์) ยังเก็บบน Drive ตามปกติเพื่อให้ข้อมูลอยู่ถาวรหลัง runtime หลุด

In [ ]:
import os

RUN_NAME   = 'stray_model_v4'
DATA_DIR   = '/content/stray_dataset'               # ดิสก์ของ Colab เอง — เร็วกว่า Drive มากตอนแตกไฟล์รูปจำนวนมาก
SAVE_PATH  = f'/content/drive/MyDrive/{RUN_NAME}'    # ผลลัพธ์สุดท้าย (โมเดล/กราฟ/ตาราง) เก็บบน Drive ให้อยู่ถาวร
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)   # กันโฟลเดอร์ยังไม่มี (จะถูกเติมข้อมูลใน cell ถัดไป)

IMG_SIZE    = 300
BATCH_SIZE  = 32
SEED        = 42

# oversampling: คลาสไหนน้อยกว่านี้ % ของคลาสที่เยอะสุด จะถูกสุ่มซ้ำเพิ่มจนถึงเพดานนี้ (ไม่ทำให้เท่ากันเป๊ะ)
OVERSAMPLE_CAP_RATIO = 0.5

print("RUN_NAME :", RUN_NAME)
print("DATA_DIR :", DATA_DIR)
print("SAVE_PATH:", SAVE_PATH)
print("มีโฟลเดอร์สายพันธุ์ทั้งหมด:", len(os.listdir(DATA_DIR)), "สายพันธุ์ (ปกติจะเป็น 0 ตอนนี้ เพราะยังไม่รัน cell เตรียมข้อมูล)")


## Cell 4 — เตรียมข้อมูล: ดาวน์โหลด Oxford-IIIT เฉพาะสายพันธุ์ที่เลือก + แตกไฟล์สายพันธุ์ที่เทรนเพิ่มเอง

**ก่อนรัน cell นี้ ต้องทำ 1 อย่างก่อน:** ที่เครื่องคอมพิวเตอร์ ไปที่โฟลเดอร์โปรเจกต์ เลือก 13 โฟลเดอร์นี้พร้อมกัน —
`calico, domestic_shorthair, khao_manee, konja, korat, labrador_retriever, siberian_husky, suphalak, thai_bangkaew, thai_mixed_breed_dog, thai_ridgeback, tortoiseshell, golden_retriever`
— คลิกขวา → Send to → Compressed (zipped) folder ตั้งชื่อไฟล์ `stray_custom_breeds.zip` แล้วอัปโหลดไปที่ Google Drive (ไว้ที่ `MyDrive/` เลย หรือแก้ path ในโค้ดให้ตรงกับที่วางไว้)

**ควรได้:** ตอนรันครั้งแรกจะเห็นข้อความ "กำลังดาวน์โหลด Oxford-IIIT..." (ไฟล์ ~800MB ใช้เวลาไม่กี่นาที) ตามด้วย "แตกไฟล์ Oxford-IIIT (เฉพาะ 18 สายพันธุ์ที่เก็บ) แล้ว xxxx รูป" และ "แตกไฟล์สายพันธุ์ที่เทรนเพิ่มเองสำเร็จ" — เพราะ `DATA_DIR` เป็นดิสก์ของ Colab เอง (ไม่ใช่ Drive) ขั้นแตกไฟล์ทั้งสองส่วนควรใช้เวลาแค่ไม่กี่วินาทีถึงไม่กี่นาที **ถ้าค้างเกิน 10 นาทีที่ขั้นแตกไฟล์ (ไม่ใช่ตอนดาวน์โหลด) แปลว่ามีอะไรผิดปกติ ให้ interrupt แล้วเช็ค `DATA_DIR` ว่าเผลอตั้งเป็น path บน Drive อยู่หรือเปล่า** บรรทัดสุดท้ายต้องนับได้ **31 สายพันธุ์** (18 จาก Oxford + 13 ที่เทรนเพิ่มเอง) ถ้าน้อยกว่านี้ แปลว่ายังอัปโหลด/วาง zip ไม่ถูกที่ — เช็ค path ของ `CUSTOM_ZIP` ให้ตรงกับที่อัปโหลดไว้จริง รันครั้งต่อไปจะเร็วขึ้นมาก เพราะ cell นี้ข้ามการดาวน์โหลด/แตกไฟล์ซ้ำถ้าตรวจแล้วว่ามีครบแล้ว

In [ ]:
import os, re, shutil, tarfile, zipfile, urllib.request
from pathlib import Path

os.makedirs(DATA_DIR, exist_ok=True)

# ---- 1) Oxford-IIIT Pet Dataset: ดาวน์โหลดเฉพาะครั้งแรก แล้วกรองเก็บเฉพาะสายพันธุ์ที่เลือกไว้ ----
OXFORD_KEEP = {
    # หมา (11)
    'beagle', 'pug', 'english_cocker_spaniel', 'american_bulldog',
    'american_pit_bull_terrier', 'pomeranian', 'chihuahua', 'shiba_inu',
    'yorkshire_terrier', 'havanese', 'japanese_chin',
    # แมว (7)
    'british_shorthair', 'persian', 'birman', 'ragdoll', 'siamese',
    'maine_coon', 'bengal',
}

oxford_tar = '/content/oxford_images.tar.gz'
if not os.path.exists(oxford_tar):
    print("กำลังดาวน์โหลด Oxford-IIIT Pet Dataset (~800MB, รอสักครู่)...")
    urllib.request.urlretrieve(
        'https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz', oxford_tar
    )
    print("ดาวน์โหลดเสร็จ")
else:
    print("เจอไฟล์ดาวน์โหลดไว้แล้ว ข้ามขั้นตอนดาวน์โหลด")

# ตรวจว่ามีสายพันธุ์ Oxford ครบแล้วหรือยัง (ข้ามถ้าทำไปแล้วรอบก่อน ประหยัดเวลา)
มีครบแล้ว = all((Path(DATA_DIR) / b).is_dir() and any((Path(DATA_DIR) / b).iterdir()) for b in OXFORD_KEEP)
if มีครบแล้ว:
    print("มีโฟลเดอร์ Oxford ครบทั้ง", len(OXFORD_KEEP), "สายพันธุ์แล้ว ข้ามการแตกไฟล์")
else:
    pattern = re.compile(r'^images/(.+)_\d+\.(jpg|jpeg|png)$', re.IGNORECASE)
    คัดลอกแล้ว = 0
    with tarfile.open(oxford_tar, 'r:gz') as tar:
        for member in tar.getmembers():
            m = pattern.match(member.name)
            if not m:
                continue
            breed = m.group(1).lower()
            if breed not in OXFORD_KEEP:
                continue
            breed_dir = Path(DATA_DIR) / breed
            breed_dir.mkdir(exist_ok=True)
            fileobj = tar.extractfile(member)
            if fileobj is None:
                continue
            out_path = breed_dir / Path(member.name).name
            with open(out_path, 'wb') as f:
                shutil.copyfileobj(fileobj, f)
            คัดลอกแล้ว += 1
    print(f"แตกไฟล์ Oxford-IIIT (เฉพาะ {len(OXFORD_KEEP)} สายพันธุ์ที่เก็บ) แล้ว {คัดลอกแล้ว} รูป")

# ---- 2) สายพันธุ์ที่เทรนเพิ่มเอง: แตกจาก zip ที่อัปโหลดไว้ใน Drive ----
CUSTOM_ZIP = '/content/drive/MyDrive/stray_custom_breeds.zip'  # <-- แก้ path ให้ตรงกับที่อัปโหลดไว้จริง

if os.path.exists(CUSTOM_ZIP):
    with zipfile.ZipFile(CUSTOM_ZIP, 'r') as z:
        z.extractall(DATA_DIR)
    print("แตกไฟล์สายพันธุ์ที่เทรนเพิ่มเองสำเร็จ")
else:
    print(f"\u26a0\ufe0f ไม่พบไฟล์ {CUSTOM_ZIP} — อัปโหลด zip ไปที่ Drive ก่อน แล้วรัน cell นี้ใหม่")

print("\nสายพันธุ์ทั้งหมดใน DATA_DIR ตอนนี้:", len(os.listdir(DATA_DIR)), "สายพันธุ์")
print(sorted(os.listdir(DATA_DIR)))


## Cell 5 — สแกนไฟล์ทั้งหมด → สร้างตารางรายการรูป (filepath, label)

**ควรได้:** ตาราง (DataFrame) แสดง 5 แถวแรก มีคอลัมน์ `filepath` กับ `label` และบรรทัดสุดท้ายบอกจำนวนรูปรวมทั้งหมด (ควรเป็นหลักพันขึ้นไปถ้า dataset สมบูรณ์)

In [ ]:
import pandas as pd
from pathlib import Path

CLASS_NAMES = sorted(os.listdir(DATA_DIR))
records = []
for label in CLASS_NAMES:
    folder = Path(DATA_DIR) / label
    if not folder.is_dir():
        continue
    for f in folder.glob('*'):
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'):
            records.append({'filepath': str(f), 'label': label})

df = pd.DataFrame(records)
print("จำนวนรูปทั้งหมด:", len(df))
print("จำนวนสายพันธุ์:", df['label'].nunique())
df.head()


## Cell 6 — ดูการกระจายตัวของข้อมูลต่อสายพันธุ์ (ก่อน oversample)

**ควรได้:** กราฟแท่งแสดงจำนวนรูปต่อสายพันธุ์ 47 แท่ง (หรือตามจำนวนสายพันธุ์จริง) เรียงจากมากไปน้อย ใช้ดูว่าสายพันธุ์ไหนข้อมูลน้อยเป็นพิเศษ (ตัวที่เป็นปัญหาเดิมคือ american_pit_bull_terrier ควรเห็นแท่งมันสั้นกว่าชาวบ้าน)

In [ ]:
import matplotlib.pyplot as plt

counts = df['label'].value_counts()

plt.figure(figsize=(14, 6))
counts.plot(kind='bar')
plt.title('จำนวนรูปต่อสายพันธุ์ (ก่อน oversample)')
plt.ylabel('จำนวนรูป')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

print("สายพันธุ์ที่มีรูปน้อยที่สุด 5 อันดับ:")
print(counts.tail(5))


## Cell 7 — แบ่ง Train / Validation / Test (แบบ stratified คงสัดส่วนแต่ละสายพันธุ์)

**ควรได้:** พิมพ์จำนวนรูปของ train/val/test ออกมา 3 บรรทัด (train ควรเยอะสุด ~70%, val/test ~15% เท่าๆกัน) รวมกันแล้วต้องเท่ากับจำนวนรูปทั้งหมดจาก Cell 3

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED
)

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))
print("รวม  :", len(train_df) + len(val_df) + len(test_df), "(ต้องเท่ากับ Cell 3)")


## Cell 8 — Oversample เฉพาะ Train set (เบาๆ ไม่ทำให้เท่ากันเป๊ะ)

**ควรได้:** กราฟแท่งใหม่หลัง oversample เทียบกับ Cell 4 — แท่งของสายพันธุ์ที่เคยสั้นมากจะสูงขึ้น (แต่ไม่เท่าสายพันธุ์ที่เยอะสุด เพราะจำกัดไว้ที่ `OVERSAMPLE_CAP_RATIO`) พิมพ์จำนวนรูป train ใหม่ (ควรมากกว่าค่าเดิมใน Cell 5 เล็กน้อยถึงปานกลาง)

In [ ]:
max_count = train_df['label'].value_counts().max()
target_count = int(max_count * OVERSAMPLE_CAP_RATIO)

oversampled_parts = []
for label, group in train_df.groupby('label'):
    if len(group) < target_count:
        extra = group.sample(target_count - len(group), replace=True, random_state=SEED)
        oversampled_parts.append(pd.concat([group, extra]))
    else:
        oversampled_parts.append(group)

train_df_os = pd.concat(oversampled_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Train เดิม:", len(train_df), "-> Train หลัง oversample:", len(train_df_os))

plt.figure(figsize=(14, 6))
train_df_os['label'].value_counts().plot(kind='bar')
plt.title('จำนวนรูปต่อสายพันธุ์ (Train, หลัง oversample)')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## Cell 9 — คำนวณ Class Weight (ใช้คู่กับ oversampling)

**ควรได้:** พิมพ์ dict ของ class weight ออกมา ตัวเลขของสายพันธุ์ที่มีรูปน้อยควร > 1.0 (ยิ่งน้อยยิ่งค่ามาก) ส่วนสายพันธุ์ที่มีรูปเยอะควรใกล้ 1.0 หรือต่ำกว่า

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
y_train_idx = train_df_os['label'].map(label_to_idx).values

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(CLASS_NAMES)),
    y=y_train_idx
)
class_weight_dict = {i: w for i, w in enumerate(weights)}

# โชว์ 5 ตัวที่ weight สูงสุด (สายพันธุ์หายากสุด) กับ 5 ตัวต่ำสุด
sorted_w = sorted(class_weight_dict.items(), key=lambda x: -x[1])
print("Class weight สูงสุด 5 อันดับ (หายากสุด):")
for i, w in sorted_w[:5]:
    print(f"  {CLASS_NAMES[i]}: {w:.2f}")
print("Class weight ต่ำสุด 5 อันดับ:")
for i, w in sorted_w[-5:]:
    print(f"  {CLASS_NAMES[i]}: {w:.2f}")


## Cell 10 — สร้าง tf.data pipeline (letterbox resize + augmentation)

**ควรได้:** ไม่มี error ตอนสร้างฟังก์ชัน (cell นี้แค่นิยามฟังก์ชัน ยังไม่เห็นผลลัพธ์ภาพ — จะเห็นใน Cell 10) จุดสำคัญ: ใช้ `tf.image.resize_with_pad` (letterbox) ให้ตรงกับที่ `main.py` ใช้ตอน inference จริงบนเว็บ ไม่ใช้วิธีบีบภาพ (squash)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize_with_pad(img, IMG_SIZE, IMG_SIZE)  # letterbox — ต้องตรงกับ main.py
    img = tf.cast(img, tf.float32)
    return img, label

def build_dataset(dataframe, shuffle=False, batch_size=BATCH_SIZE):
    labels_idx = dataframe['label'].map(label_to_idx).values
    labels_onehot = tf.keras.utils.to_categorical(labels_idx, num_classes=len(CLASS_NAMES)).astype('float32')
    ds = tf.data.Dataset.from_tensor_slices((dataframe['filepath'].values, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds_raw = build_dataset(train_df_os, shuffle=True)
val_ds       = build_dataset(val_df)
test_ds      = build_dataset(test_df)
print("สร้าง pipeline สำเร็จ")


## Cell 11 — ตั้งค่า mixed precision + สร้างชั้น Augmentation

**ควรได้:** ข้อความ `Compute dtype: float16` (หรือใกล้เคียง) ไม่มี error — จุดสำคัญ (บั๊กที่เจอใน v3): ต้องใส่ `dtype='float32'` ในแต่ละ augmentation layer แยกกัน ห้ามใส่ที่ `Sequential(...)` เพราะจะ error `unexpected keyword argument 'dtype'`

In [ ]:
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')
print("Compute dtype:", mixed_precision.global_policy().compute_dtype)

augment_layer = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', dtype='float32'),
    tf.keras.layers.RandomRotation(0.08, dtype='float32'),
    tf.keras.layers.RandomZoom(0.1, dtype='float32'),
    tf.keras.layers.RandomTranslation(0.05, 0.05, dtype='float32'),
    tf.keras.layers.RandomContrast(0.1, dtype='float32'),
], name='augment_layer')

print("สร้าง augmentation layer สำเร็จ")


## Cell 12 — MixUp (เฉพาะ Phase 1 เท่านั้น) + ประกอบ pipeline สุดท้าย

**ควรได้:** เห็นภาพตัวอย่าง 6 รูปจาก train set ที่ผ่าน augmentation + MixUp แล้ว — ภาพจะดู "ซ้อนกันจางๆ" บางรูป (2 ตัวสัตว์เบลอปนกัน) นั่นคือ MixUp ทำงานถูกต้อง ไม่ใช่บั๊ก (เหมือนที่เจอตอน v3)

In [ ]:
def mixup(images, labels, alpha=0.2):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.float32)
    batch_size = tf.shape(images)[0]
    gamma1 = tf.random.gamma([], alpha, dtype=tf.float32)
    gamma2 = tf.random.gamma([], alpha, dtype=tf.float32)
    lam = gamma1 / (gamma1 + gamma2)   # สุ่มแบบ Beta(alpha, alpha) ด้วย 2 ค่า Gamma แทน API เก่าที่คืน float64
    idx = tf.random.shuffle(tf.range(batch_size))

    images_mixed = lam * images + (1 - lam) * tf.gather(images, idx)
    labels_mixed = lam * labels + (1 - lam) * tf.gather(labels, idx)
    return images_mixed, labels_mixed

def prepare_phase1(images, labels):
    images = augment_layer(images, training=True)
    images, labels = mixup(images, labels, alpha=0.2)
    return images, labels

def prepare_phase23(images, labels):
    # Phase 2-3: augment ปกติ แต่ "ไม่ใช้ MixUp" — จุดที่เปลี่ยนจาก v3 เพื่อไม่ให้เบลอ
    # จุดต่างเล็กๆระหว่างสายพันธุ์หน้าคล้ายกัน (เช่น Ragdoll/Birman) ตอน fine-tune ชั้นลึก
    images = augment_layer(images, training=True)
    return images, labels

train_ds_phase1 = train_ds_raw.map(prepare_phase1, num_parallel_calls=AUTOTUNE)
train_ds_phase23 = train_ds_raw.map(prepare_phase23, num_parallel_calls=AUTOTUNE)

# ดูตัวอย่างภาพจาก phase1 (มี MixUp)
sample_images, sample_labels = next(iter(train_ds_phase1))
plt.figure(figsize=(14, 8))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(sample_images[i].numpy().astype('uint8'))
    top2 = np.argsort(sample_labels[i].numpy())[-2:][::-1]
    plt.title(f"{CLASS_NAMES[top2[0]]} + {CLASS_NAMES[top2[1]]}", fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()


## Cell 13 — สร้างโมเดล (EfficientNetB0 + หัวใหม่)

**ควรได้:** `model.summary()` แสดงออกมา ต้องเห็นว่า backbone (efficientnetb0) เป็น non-trainable ตอนเริ่ม (Phase 1) และมี layer ใหม่ต่อท้าย (GlobalAveragePooling, Dropout, Dense) จำนวนพารามิเตอร์รวมหลักล้าน (Trainable params ควรเป็นแค่หลักหมื่น-แสนในตอนนี้ เพราะ backbone ยัง freeze)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    include_top=False, weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax', dtype='float32')(x)

model = models.Model(inputs, outputs)
model.summary()


## Cell 14 — Macro-F1 Callback + Focal Loss + Early Stopping (ใช้ร่วมกันทุก Phase)

**ควรได้:** ไม่มี error (แค่นิยาม callback/loss function) — ส่วนนี้เพิ่ม `EarlyStopping` เข้ามาใหม่จาก v3 (ของเดิมไม่มี) เพื่อกัน Phase 3 เทรนเลยจุดที่ดีที่สุดไปแล้ว

In [ ]:
from sklearn.metrics import f1_score

class MacroF1Callback(tf.keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__()
        self.val_data = val_data
        self.best_f1 = 0.0

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        for imgs, labels in self.val_data:
            preds = self.model.predict(imgs, verbose=0)
            y_true.extend(np.argmax(labels.numpy(), axis=1))
            y_pred.extend(np.argmax(preds, axis=1))
        macro_f1 = f1_score(y_true, y_pred, average='macro')
        logs['val_macro_f1'] = macro_f1
        print(f"  -> val_macro_f1: {macro_f1:.4f}")
        if macro_f1 > self.best_f1:
            self.best_f1 = macro_f1

focal_loss = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=1.5, label_smoothing=0.05
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_macro_f1', mode='max', patience=5,
    restore_best_weights=True, verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
)

macro_f1_cb = MacroF1Callback(val_ds)
print("เตรียม loss/callback สำเร็จ")


## Cell 15 — เทรน Phase 1: หัวโมเดลอย่างเดียว (backbone freeze, มี MixUp)

**ควรได้:** เทรน 10 epoch เห็น accuracy เพิ่มขึ้นเรื่อยๆ (เริ่มจากประมาณ 5-15% ไปจนถึง 40-60% ช่วงท้าย Phase 1 ถือว่าปกติ เพราะยังไม่ fine-tune backbone) และมีบรรทัด `val_macro_f1` โผล่ท้ายทุก epoch จาก callback ของเรา**หมายเหตุ:** ใช้ `train_ds_phase1` (มี MixUp) ไม่ใช่ `train_ds_phase23`

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=focal_loss,
    metrics=['accuracy']
)

EPOCHS_PHASE1 = 10

history1 = model.fit(
    train_ds_phase1,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weight_dict,
    callbacks=[macro_f1_cb, reduce_lr]
)


## Cell 16 — Phase 2: ปลด freeze backbone 60 ชั้นสุดท้าย (ไม่มี MixUp แล้ว)

**ควรได้:** `model.summary()` (หรือ print) แสดงว่ามี Trainable params เพิ่มขึ้นมากจาก Phase 1 (จากหลักหมื่นเป็นหลักล้าน) เพราะปลด freeze บางส่วนของ backbone แล้ว

In [ ]:
base_model.trainable = True

# freeze ทุกชั้นยกเว้น 60 ชั้นสุดท้ายของ backbone
for layer in base_model.layers[:-60]:
    layer.trainable = False

trainable_count = sum([1 for l in base_model.layers if l.trainable])
print(f"ปลด freeze แล้ว {trainable_count} / {len(base_model.layers)} ชั้นใน backbone")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # LR ต่ำลง เพราะ fine-tune แล้ว
    loss=focal_loss,
    metrics=['accuracy']
)


## Cell 17 — เทรน Phase 2 (unfreeze -60 ชั้น, ไม่มี MixUp)

**ควรได้:** เทรนต่อจาก epoch 11-25 (15 epoch) accuracy ควรขยับขึ้นกว่าท้าย Phase 1 พอสมควร (เช่นจาก ~55% ไป ~70-80%) ถ้า `val_macro_f1` เริ่มนิ่งหรือลดหลายรอบติด — `EarlyStopping` อาจตัดจบก่อนครบ 15 epoch ซึ่งเป็นเรื่องปกติ ไม่ใช่ error

In [ ]:
EPOCHS_PHASE2 = 25  # ค่า epochs สะสม (absolute) ไม่ใช่จำนวนรอบเพิ่ม — initial_epoch มาจาก Phase 1

history2 = model.fit(
    train_ds_phase23,
    validation_data=val_ds,
    initial_epoch=EPOCHS_PHASE1,
    epochs=EPOCHS_PHASE2,
    class_weight=class_weight_dict,
    callbacks=[macro_f1_cb, early_stop, reduce_lr]
)


## Cell 18 — Phase 3: ปลด freeze backbone ทั้งหมด

**ควรได้:** print บอกว่า backbone ทุกชั้น trainable แล้ว (เทียบ Cell 14 ที่ปลดแค่ 60 ชั้น)

In [ ]:
for layer in base_model.layers:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # LR ต่ำสุด เพราะปลด freeze หมดแล้ว
    loss=focal_loss,
    metrics=['accuracy']
)
print("ปลด freeze backbone ทั้งหมดแล้ว — เทรนได้ทุกชั้น")


## Cell 19 — เทรน Phase 3 (fine-tune เต็มทั้งโมเดล, ไม่มี MixUp)

**ควรได้:** เทรนต่อจากจุดที่ Phase 2 จบจริง (ใช้จำนวน epoch จริงที่ Phase 2 รันได้ ไม่ใช่ 25 เป๊ะ เผื่อ EarlyStopping ตัดก่อน) accuracy ควรขึ้นไปแตะช่วง 80-90%+ ถ้า `EarlyStopping` ทำงาน จะเห็นข้อความ `Restoring model weights from the end of the best epoch` ก่อนเทรนจบ ซึ่งเป็นเรื่องดี ไม่ใช่ error

In [ ]:
# epochs ต้องเป็นค่าสะสม (absolute) — คำนวณจาก epoch จริงที่ Phase 1+2 รันไปแล้ว
initial_epoch_phase3 = len(history1.history['accuracy']) + len(history2.history['accuracy'])
EPOCHS_PHASE3 = initial_epoch_phase3 + 20

history3 = model.fit(
    train_ds_phase23,
    validation_data=val_ds,
    initial_epoch=initial_epoch_phase3,
    epochs=EPOCHS_PHASE3,
    class_weight=class_weight_dict,
    callbacks=[macro_f1_cb, early_stop, reduce_lr]
)


## Cell 20 — รวมประวัติการเทรนทั้ง 3 Phase + วาดกราฟ Accuracy / Loss ต่อ epoch

**ควรได้:** กราฟ 2 รูป (Accuracy per epoch และ Loss per epoch) แต่ละรูปมี 2 เส้น (train / val) ยาวต่อเนื่องตลอดทุก epoch ของทั้ง 3 phase มีเส้นประแนวตั้ง 2 เส้นคั่นบอกจุดเริ่ม Phase 2 และ Phase 3 เส้น accuracy ควรค่อยๆไต่ขึ้น เส้น loss ควรค่อยๆลดลง (อาจมีสะดุดเล็กน้อยตรงรอยต่อ phase ซึ่งปกติ)

In [ ]:
def รวมค่า(key):
    return history1.history.get(key, []) + history2.history.get(key, []) + history3.history.get(key, [])

acc      = รวมค่า('accuracy')
val_acc  = รวมค่า('val_accuracy')
loss     = รวมค่า('loss')
val_loss = รวมค่า('val_loss')

epoch_range = range(1, len(acc) + 1)
phase2_start = len(history1.history['accuracy'])
phase3_start = phase2_start + len(history2.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(epoch_range, acc, label='Train Accuracy')
axes[0].plot(epoch_range, val_acc, label='Val Accuracy')
axes[0].axvline(phase2_start, color='gray', linestyle='--', alpha=0.6, label='เริ่ม Phase 2')
axes[0].axvline(phase3_start, color='black', linestyle='--', alpha=0.6, label='เริ่ม Phase 3')
axes[0].set_title('Accuracy per Epoch (รวมทุก Phase)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(epoch_range, loss, label='Train Loss')
axes[1].plot(epoch_range, val_loss, label='Val Loss')
axes[1].axvline(phase2_start, color='gray', linestyle='--', alpha=0.6, label='เริ่ม Phase 2')
axes[1].axvline(phase3_start, color='black', linestyle='--', alpha=0.6, label='เริ่ม Phase 3')
axes[1].set_title('Loss per Epoch (รวมทุก Phase)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/accuracy_loss_per_epoch.png', dpi=150)
plt.show()
print(f"บันทึกกราฟไว้ที่ {SAVE_PATH}/accuracy_loss_per_epoch.png")


## Cell 21 — ประเมินผลบน Test set: เก็บค่าจริง vs ค่าทำนาย

**ควรได้:** พิมพ์ `Test Accuracy` ออกมาเป็น % (ตัวเลขนี้คือค่าที่ควรเทียบกับ `stray_model_70_v2` เดิมและ `stray_model_v3` — เป้าหมายคือสูงกว่า v2 และไม่มีจุดถดถอยแบบ v3)

In [ ]:
y_true, y_pred, y_pred_proba = [], [], []

for imgs, labels in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
    y_pred_proba.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

test_accuracy = (y_true == y_pred).mean()
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


## Cell 22 — ตาราง Precision / Recall / F1-Score ทุกสายพันธุ์

**ควรได้:** ตาราง (DataFrame) 47 แถว (ตามจำนวนสายพันธุ์) คอลัมน์ precision/recall/f1-score/support เรียงจาก f1-score น้อยไปมาก (ตัวที่แย่สุดอยู่บนสุด) ให้เช็คว่า `american_pit_bull_terrier`, `ragdoll`, `birman` มี recall/f1 เท่าไหร่เทียบกับรอบก่อน ไฟล์ CSV จะถูกบันทึกไว้ให้ด้วย

In [ ]:
from sklearn.metrics import classification_report

report_dict = classification_report(
    y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report_dict).transpose()
report_df = report_df.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
report_df = report_df.sort_values('f1-score')

report_df.to_csv(f'{SAVE_PATH}/classification_report.csv', encoding='utf-8-sig')
print(f"บันทึกตารางไว้ที่ {SAVE_PATH}/classification_report.csv\n")

print("Macro avg F1:", report_dict['macro avg']['f1-score'])
print("Weighted avg F1:", report_dict['weighted avg']['f1-score'])
report_df


## Cell 23 — Confusion Matrix (ตัวเลขดิบ)

**ควรได้:** heatmap ขนาด 47x47 (ตามจำนวนสายพันธุ์) เส้นทแยงมุมควรมีสีเข้ม/ตัวเลขสูง (ทำนายถูก) จุดที่ต้องเพ่งดูเป็นพิเศษ: ช่อง `ragdoll` แถว ตัดกับคอลัมน์ `birman` (และกลับกัน) ควรมีตัวเลขต่ำ ถ้ายังสูงอยู่แปลว่าปัญหาจาก v3 ยังไม่หาย ต้องแก้ที่ข้อมูล (เพิ่มรูปมุมต่างๆของสองสายพันธุ์นี้) ไม่ใช่แก้ที่โค้ดเทรนอีกต่อไป

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv(f'{SAVE_PATH}/confusion_matrix_raw.csv', encoding='utf-8-sig')

plt.figure(figsize=(20, 18))
sns.heatmap(cm_df, annot=False, cmap='Blues', cbar=True)
plt.title('Confusion Matrix (ตัวเลขดิบ) — ทุกสายพันธุ์')
plt.xlabel('ทำนายว่าเป็น'); plt.ylabel('ของจริงคือ')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/confusion_matrix_raw.png', dpi=150)
plt.show()
print(f"บันทึกไฟล์ไว้ที่ {SAVE_PATH}/confusion_matrix_raw.png และ .csv")


## Cell 24 — Confusion Matrix (normalize % ต่อแถว) + สรุปคู่ที่สับสนที่สุด

**ควรได้:** heatmap เหมือน Cell 21 แต่ตัวเลขเป็น % (แต่ละแถวรวมกันได้ 100%) และตารางท้าย cell แสดง 10 คู่สายพันธุ์ที่สับสนกันมากที่สุด (นอกเส้นทแยงมุม) เรียงจากมากไปน้อย — ใช้ตารางนี้ตัดสินใจว่ารอบหน้าต้องเพิ่มข้อมูลสายพันธุ์ไหนเป็นพิเศษ

In [ ]:
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_norm_df = pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES)

plt.figure(figsize=(20, 18))
sns.heatmap(cm_norm_df, annot=False, cmap='Blues', cbar=True, vmin=0, vmax=1)
plt.title('Confusion Matrix (Normalized %) — ทุกสายพันธุ์')
plt.xlabel('ทำนายว่าเป็น'); plt.ylabel('ของจริงคือ')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/confusion_matrix_normalized.png', dpi=150)
plt.show()

# หาคู่ที่สับสนกันมากที่สุด (ไม่นับเส้นทแยงมุม)
confusions = []
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        if i != j and cm[i, j] > 0:
            confusions.append({
                'ของจริง': CLASS_NAMES[i],
                'ทายผิดเป็น': CLASS_NAMES[j],
                'จำนวนครั้ง': cm[i, j],
                '% ของจริงกลุ่มนี้': round(cm_norm[i, j] * 100, 1)
            })

confusions_df = pd.DataFrame(confusions).sort_values('จำนวนครั้ง', ascending=False)
confusions_df.to_csv(f'{SAVE_PATH}/top_confusions.csv', encoding='utf-8-sig', index=False)
print("10 คู่สายพันธุ์ที่สับสนกันมากที่สุด:")
confusions_df.head(10)


## Cell 25 — บันทึกโมเดล + แปลงเป็น ONNX + metadata.json (สำหรับ deploy)

**ควรได้:** ข้อความ `แปลง ONNX สำเร็จ` และเห็นไฟล์ `final_model.onnx` + `metadata.json` ถูกสร้างใน `SAVE_PATH` โครงสร้าง metadata.json ต้องตรงกับที่ `ai_server/main.py` อ่าน (`classes`, `breed_info`, `img_size`, `version`, `test_accuracy`) ไม่งั้น deploy แล้ว AI server จะ error ตอน parse metadata

In [ ]:
import tf2onnx
import json as jsonlib

model.save(f'{SAVE_PATH}/model.keras')

spec = (tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 3), tf.float32, name='input'),)
model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=13,
                                             output_path=f'{SAVE_PATH}/final_model.onnx')
print("แปลง ONNX สำเร็จ")

# ปรับ breed_info ตรงนี้ตามข้อมูลจริงของแต่ละสายพันธุ์ (ชื่อไทย/ประเภท/ขนาด/นิสัย)
# ถ้ามี metadata.json เดิมจาก stray_model_70_v2 อยู่แล้ว แนะนำโหลดมาแก้ต่อแทนเขียนใหม่ทั้งหมด
metadata = {
    'version': RUN_NAME,
    'classes': CLASS_NAMES,
    'img_size': IMG_SIZE,
    'test_accuracy': float(test_accuracy),
    'breed_info': {name: {'ชื่อไทย': name, 'ประเภท': 'ไม่ทราบ', 'ขนาด': 'ไม่ทราบ', 'นิสัย': 'ไม่มีข้อมูล'}
                   for name in CLASS_NAMES}
}
with open(f'{SAVE_PATH}/metadata.json', 'w', encoding='utf-8') as f:
    jsonlib.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"บันทึกทุกไฟล์ไว้ที่: {SAVE_PATH}")
print("ไฟล์ในโฟลเดอร์:", os.listdir(SAVE_PATH))
